# 数据库前传：数据都存在哪儿？

## 到现在为止，项目没有记忆

让项目拥有记忆的能力叫**持久化**（persistence）。

## 存在哪儿：四种常见存储

计算机中常见的存储位置有四类：

| 存储方式 | 特点 | 常见用途 |
| --- | --- | --- |
| 内存 | 速度快，但进程停止或断电后数据消失 | 运行中的变量、缓存、前端 state |
| 文件 | 直接写入硬盘，简单、持久 | 配置、日志、小规模数据 |
| 数据库 | 为结构化数据的增删改查设计，可筛选、排序、统计 | 业务记录、用户数据、订单等 |
| 云/对象存储 | 适合保存大文件 | 图片、视频、备份、附件 |

## 存到文件：存什么，怎么存？

假设要把用户在文字实验室分析过的文字和结果保存到文件中，首先需要决定两件事：

1. 一条记录包含什么？
2. 使用什么格式？

### 一条记录包含什么？

至少需要五个字段：

| 字段 | 含义 |
| --- | --- |
| `text` | 用户输入的原文 |
| `score` | 情感分数 |
| `label` | 情感结论 |
| `pinyin` | 拼音 |
| `created_at` | 分析发生的时间 |

### 使用 JSON

JSON 能用字段名固定结构，也能正确保存字符串内部的逗号：

~~~json
[
  {
    "text": "我喜欢，真的喜欢",
    "score": 0.96,
    "label": "偏积极",
    "pinyin": "wǒ xǐ huān ， zhēn de xǐ huān",
    "created_at": "2026-07-04T07:31:12+00:00"
  }
]
~~~

## 时区这个坑

直接调用 `datetime.now()` 得到的是当前机器的本地时间，而且可能没有时区信息。但服务器可能位于任何国家，用户也可能来自不同时区。

业界常见做法是：

> 存储时统一使用 UTC；显示给用户时，再转换成用户所在时区。

UTC 是全球统一、无歧义的时间基准。北京时间是 UTC+8。

Python 的 `datetime` 属于标准库，不需要额外安装：

~~~pycon
>>> from datetime import datetime, timezone
>>> datetime.now(timezone.utc).isoformat(timespec="seconds")
'2026-07-04T07:30:00+00:00'
~~~

末尾的 `+00:00` 明确表示这是 UTC 时间。存储层使用 UTC，展示层负责转换。

## 写入文件：读档和存档

回到 `main.py`，在顶部加入两个标准库导入：

~~~python
import json
from datetime import datetime, timezone
~~~

然后增加历史文件路径，以及读档、存档函数：

~~~python
HISTORY_FILE = "history.json"


def load_history():
    try:
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return []


def save_record(record):
    records = load_history()
    records.append(record)

    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
~~~

### 两个新面孔

`with open(...) as f`：

- 打开文件供代码读取或写入。
- `with` 代码块结束后，文件会被自动关闭。
- 这是 Python 操作文件的常用固定搭配。

`try / except FileNotFoundError`：

- 先尝试读取 `history.json`。
- 第一次运行时文件还不存在，会出现 `FileNotFoundError`。
- 代码提前接住这个错误，并把历史记录当作空列表 `[]`。

`save_record` 的步骤非常直白：

~~~text
读出全部记录 → 追加一条 → 把整个列表写回文件
~~~

`ensure_ascii=False` 让中文原样保存，`indent=2` 增加缩进，方便人直接查看文件。

## 分析完成后顺手存档

修改 `analyze`，给结果增加 UTC 时间戳，并在返回前保存记录：

~~~python
@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    text = req.text
    score = round(SnowNLP(text).sentiments, 2)

    result = {
        "text": text,
        "score": score,
        "label": score_label(score),
        "pinyin": " ".join(lazy_pinyin(text, style=Style.TONE)),
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

    save_record(result)
    return result
~~~

每次请求现在会经历：

~~~text
接收文字
  → 计算拼音和情感
  → 生成 UTC 时间
  → 写入 history.json
  → 把同一份结果返回给调用方
~~~

## 增加读取历史的接口

新建一个只读接口 `GET /api/history`。先使用最直白的写法：

~~~python
@app.get("/api/history")
def history():
    records = load_history()
    return records
~~~

启动后端：

~~~bash
uv run fastapi dev
~~~

新开终端测试：

~~~bash
curl http://localhost:8000/api/history
~~~

第一次可能返回：

~~~json
[]
~~~

这是正常的：服务启动后还没有产生任何分析记录，`history.json` 也可能尚未创建。去文字实验室分析两三句话，再次请求历史接口，就能看到记录。

## 让最新记录排在前面，并限制数量

当前历史接口还有两个问题：

1. 文件按追加顺序保存，旧记录在前、新记录在后；查看历史时通常希望先看到最新记录。
2. 记录越来越多后，一次返回全部内容会又多又慢；页面一般只需要最近几条。

修改历史接口：

~~~python
@app.get("/api/history")
def history():
    records = load_history()
    records.reverse()
    return records[:10]
~~~

两行代码的作用：

- `records.reverse()`：就地倒转列表，让新记录排在前面。
- `records[:10]`：使用 Python 切片，只返回前 10 条。

## 用浏览器加入网址

选择优质打印

http://localhost:8000/api/history

## 见证持久化

用 VS Code 打开 `backend/history.json`，可以看到类似：

~~~json
[
  {
    "text": "今天心情不错",
    "score": 0.88,
    "label": "偏积极",
    "pinyin": "jīn tiān xīn qíng bù cuò",
    "created_at": "2026-07-04T07:51:03+00:00"
  }
]
~~~

每次分析的结果都清楚地写在文件中：中文原样、字段明确、格式整齐。这就是**数据落盘**。

接下来做一次验证：

1. 按 `Ctrl + C` 停止后端。
2. 重新启动后端。
3. 再次调用 `/api/history`。

记录仍然存在，一条也没有少。

## 文件写入的隐患

- 存一条，重写一整份
- 脏写：同时写入会互相覆盖
- 写一半失败，整份文件都无法解析

## 文件读取的隐患

- 只要十条，缺读入全部
- 脏读：读到别人还未写完的状态

## 文件方案的四处隐患可以汇总为：

| 问题 | 类型 |
| --- | --- |
| 只取 10 条，却读取全部 | 读取效率 |
| 只存 1 条，却重写整份 | 写入效率 |
| 并发时出现脏写、脏读 | 并发一致性 |
| 写到一半失败，一坏全坏 | 健壮性与恢复 |

## 数据库为什么会出现？

文件方案暴露的问题，尤其是“多人同时读写不能出乱子”，需要专门的机制处理。

其中一个重要概念叫**事务**（transaction）。事务帮助一组数据操作以可靠、完整的方式执行，避免其他操作看到不一致的中间状态，也能在失败时进行处理。

提供结构化查询、索引、并发控制和事务等能力的软件，就是数据库。